# Day 3 — FAISS & Vector Indexes

---

Yesterday's naive search worked for 8 documents. Today we scale to **hundreds of thousands** without breaking a sweat, using an **ANN index** — Approximate Nearest Neighbor.

By the end of 75 minutes you'll:

1. Understand **ANN** in plain English
2. Build and query a **FAISS** index
3. Save and load an index to disk (so you don't re-embed every restart)
4. Benchmark it against yesterday's naive search


## 1. What's an ANN index, in plain English?

Yesterday we compared the query against **every single document**. That's exact but slow.

An **ANN index** is a smart data structure that says: "I don't need to check every document — I'll organize them ahead of time so I can skip most comparisons."

- **A** = Approximate → occasionally misses the *absolute* best match, but usually gets a near-best one
- **NN** = Nearest Neighbor → finds the closest vectors

**Real-world analogy:** Finding a friend in a stadium of 50,000 people.

- **Naive search**: check every seat, one at a time. Guaranteed to find them. Takes forever.
- **ANN search**: use the seat number to jump to their section, then their row. Almost always finds them instantly. Rarely, if you got the wrong section, you might miss — but it's close enough.

**Trade-off:** you sacrifice a tiny amount of accuracy for a **massive** speedup. In practice, ANN recovers 95–99% of the "true" best results and is 100–1000× faster.


## 2. FAISS — Facebook's vector index library

**FAISS** is the most popular library for building ANN indexes on your own machine. It's free, fast, and runs on your CPU.

You don't need to memorize the algorithms inside. Just know these three index types exist:

| Index type | What it does | When to use |
|---|---|---|
| `IndexFlatIP` | Naive (exact) search | Small datasets (<10k). Baseline. |
| `IndexHNSWFlat` | Graph-based ANN | Default choice up to a few million vectors. Fast, accurate. |
| `IndexIVFFlat` | Cluster-based ANN | Huge datasets (100M+). Uses less memory. |

For this course we'll use **`IndexHNSWFlat`** — it's the sweet spot for real projects.


In [ ]:
!pip install faiss-cpu sentence-transformers --quiet

## 3. Build your first FAISS index


In [ ]:
import numpy as np
import faiss
from sentence_transformers import SentenceTransformer

model = SentenceTransformer("all-MiniLM-L6-v2")

docs = [
    "Python is a popular programming language.",
    "The Eiffel Tower is located in Paris, France.",
    "Machine learning models are trained on data.",
    "Croissants are a famous French pastry.",
    "FastAPI is a modern Python web framework.",
    "Neural networks are inspired by the human brain.",
    "The Louvre museum houses the Mona Lisa.",
    "Django is another Python web framework.",
]

# 1. Embed & normalize (cosine similarity works via dot product on normalized vectors)
vectors = model.encode(docs).astype("float32")
faiss.normalize_L2(vectors)

# 2. Build an HNSW index. Second arg is "M" — connections per node. 32 is a solid default.
index = faiss.IndexHNSWFlat(vectors.shape[1], 32)
index.add(vectors)
print(f"Index built. Total vectors: {index.ntotal}")


**Two important lines:**

- `faiss.normalize_L2(vectors)` — normalizes every vector to length 1. This lets us use dot product as cosine similarity. (You'll do this in every FAISS project — copy-paste and move on.)
- `IndexHNSWFlat(dim, 32)` — creates an HNSW graph. `32` is the number of connections each node has. Bigger = more accurate but slower to build.


## 4. Search the index


In [ ]:
def search(query: str, top_k: int = 3):
    q = model.encode([query]).astype("float32")
    faiss.normalize_L2(q)
    scores, ids = index.search(q, top_k)
    return [(docs[i], float(s)) for s, i in zip(scores[0], ids[0])]

for hit, score in search("web development in Python"):
    print(f"  {score:.3f}  {hit}")


Same results as yesterday's naive search. The API is a little different — `index.search()` returns two arrays: `scores` and `ids`. The `ids` map back to your original `docs` list.


## 5. Save & load — don't re-embed every restart

Embedding 100k docs takes minutes. You don't want to redo it every time your app starts. FAISS lets you save the index to a file.


In [ ]:
import os

faiss.write_index(index, "notes.faiss")
print("Saved. File size:", os.path.getsize("notes.faiss"), "bytes")

# Later, in another script or after a restart:
loaded = faiss.read_index("notes.faiss")
print("Loaded. Vectors in index:", loaded.ntotal)


**Important:** the index file only stores vectors + IDs. It does **not** store your original documents. In a real app you'd keep the docs in a database (or JSON file) and use the IDs from FAISS to look them up.


## 6. Naive vs FAISS — the speedup

Let's index 50,000 fake vectors and compare.


In [ ]:
import time
import numpy as np
import faiss

N, dim = 50_000, 384
fake_docs = np.random.randn(N, dim).astype("float32")
faiss.normalize_L2(fake_docs)
query = np.random.randn(1, dim).astype("float32")
faiss.normalize_L2(query)

# --- Naive: compare against every doc ---
start = time.time()
scores = fake_docs @ query[0]
np.argsort(-scores)[:10]
naive_ms = (time.time() - start) * 1000

# --- FAISS HNSW ---
idx = faiss.IndexHNSWFlat(dim, 32)
idx.add(fake_docs)
start = time.time()
idx.search(query, 10)
faiss_ms = (time.time() - start) * 1000

print(f"Naive: {naive_ms:7.2f} ms")
print(f"FAISS: {faiss_ms:7.2f} ms")
print(f"Speedup: ~{naive_ms/faiss_ms:.0f}x")


On 50k vectors FAISS is already ~50× faster. Try `N = 500_000` and you'll see the gap widen dramatically.


## Recap

- **ANN** = give up a tiny bit of accuracy for a huge speedup.
- **FAISS** is the go-to library. Use `IndexHNSWFlat` for most projects.
- Always **normalize** vectors before adding, then dot product = cosine similarity.
- **Save the index to disk** so you don't re-embed on every restart.
- Store your original documents separately — FAISS only stores vectors.
- **Next class:** vector databases like ChromaDB — same idea, but with metadata, filtering, and no manual file management.
